In [1]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd 

import itertools
from collections import defaultdict

from scripts.ss_graphgen import *

In [2]:
dtx = pd.read_csv("../data/student_math.csv")

print("Original number of rows: ", len(dtx))
print(" ")

dt = dtx.drop_duplicates().reset_index(drop=True)
dt


Original number of rows:  395
 


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,pass
0,0,0,18,1,0,0,4,4,0,4,...,0,0,4,3,4,1,1,3,6,0
1,0,0,17,1,0,1,1,1,0,2,...,1,0,5,3,3,1,1,3,4,0
2,0,0,15,1,1,1,1,1,0,2,...,1,0,4,3,2,2,3,3,10,0
3,0,0,15,1,0,1,4,2,1,3,...,1,1,3,2,2,1,1,5,2,1
4,0,0,16,1,0,1,3,3,2,2,...,0,0,4,3,2,1,2,5,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,1,1,20,1,1,0,2,2,3,3,...,0,0,5,5,4,4,5,4,11,0
391,1,1,17,1,1,1,3,1,3,3,...,1,0,2,4,5,3,4,2,3,1
392,1,1,21,0,0,1,1,1,2,2,...,0,0,5,5,3,3,3,3,3,0
393,1,1,18,0,1,1,3,2,3,2,...,1,0,4,4,1,3,4,5,0,0


In [3]:
target_column = "pass"

X = dt.drop([target_column], axis=1).values
y = dt[target_column].values
y = np.where(y == 1, 1, -1) 

s = dt["sex"].values

scaler = StandardScaler()
X = scaler.fit_transform(X)



In [4]:
##########
np.random.seed(42)
n = X.shape[0]
indices = np.random.permutation(n)
split = int(0.9 * n)
lhs_idx = indices[:split]
rhs_idx = indices[split:]

lhs_idx_negative = lhs_idx[y[lhs_idx] == -1]
X_LHS_random = X[lhs_idx_negative]
X_RHS_random = X[rhs_idx]
y_RHS_random = y[rhs_idx]

s_LHS_random = s[lhs_idx_negative]
s_RHS_random = s[rhs_idx]

xxneg = y[lhs_idx]

len(lhs_idx_negative), len(lhs_idx), len(rhs_idx), np.sum(xxneg == -1), np.sum(xxneg == 1)


(206, 355, 40, 206, 149)

In [5]:
##########
##########
edges_random, labels_random, graph_random = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                                 XRHS=X_RHS_random, 
                                                                 yRHS=y_RHS_random, 
                                                                 sLHS=s_LHS_random, 
                                                                 sRHS=s_RHS_random,                                                                
                                                                 k_min=1, 
                                                                 k_max=10)

r_edges_random = graph_random['edges']
r_labels_random = graph_random['labels']

r_graph_structure_random = {
    f"x_{x}": [f"t_{t} ({'+' if r_labels_random[t] == 1 else '-'})" for t in neighbors]
    for x, neighbors in r_edges_random.items()
}

print("Random bipartite graph")
for agent, targets in list(r_graph_structure_random.items())[:10]: 
    print(f"{agent}: {targets}")
    
    
print (" ")
print("positive ", {t for t, l in r_labels_random.items() if l == 1})
print("negative ", {t for t, l in r_labels_random.items() if l == -1})
    
graph_random['n'], graph_random['m']


Random bipartite graph
x_0: ['t_15 (-)', 't_8 (-)', 't_17 (-)', 't_12 (-)', 't_0 (+)', 't_33 (+)', 't_10 (-)', 't_39 (+)', 't_28 (+)', 't_11 (-)']
x_1: ['t_10 (-)', 't_6 (+)', 't_23 (-)', 't_29 (-)', 't_0 (+)', 't_2 (+)', 't_21 (+)', 't_38 (+)', 't_30 (-)', 't_39 (+)']
x_2: ['t_17 (-)', 't_11 (-)', 't_6 (+)', 't_34 (-)', 't_38 (+)', 't_28 (+)', 't_3 (-)', 't_1 (+)', 't_7 (+)', 't_10 (-)']
x_3: ['t_5 (-)', 't_37 (-)', 't_29 (-)', 't_24 (+)', 't_1 (+)', 't_2 (+)', 't_9 (+)', 't_22 (-)', 't_32 (-)', 't_10 (-)']
x_4: ['t_2 (+)', 't_6 (+)', 't_3 (-)', 't_9 (+)', 't_10 (-)', 't_8 (-)', 't_31 (+)', 't_35 (-)', 't_13 (+)', 't_29 (-)']
x_5: ['t_17 (-)', 't_2 (+)', 't_10 (-)', 't_28 (+)', 't_3 (-)', 't_38 (+)', 't_6 (+)', 't_36 (-)', 't_11 (-)', 't_31 (+)']
x_6: ['t_38 (+)', 't_7 (+)', 't_3 (-)', 't_28 (+)', 't_18 (+)', 't_30 (-)', 't_33 (+)', 't_11 (-)', 't_21 (+)', 't_31 (+)']
x_7: ['t_32 (-)', 't_2 (+)', 't_25 (+)', 't_0 (+)', 't_5 (-)', 't_39 (+)', 't_6 (+)', 't_24 (+)', 't_9 (+)', 't_29 (-)

(206, 39)

In [6]:
##########
edges_random, labels_random, graph_random = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                                    XRHS=X_RHS_random, 
                                                                    yRHS=y_RHS_random,
                                                                    sLHS=s_LHS_random, 
                                                                    sRHS=s_RHS_random,   
                                                                    threshold=5, 
                                                                    topk=False, 
                                                                    thresh=True)


r_edges_random = graph_random['edges']
r_labels_random = graph_random['labels']

r_graph_structure_random = {
    f"x_{x}": [f"t_{t} ({'+' if r_labels_random[t] == 1 else '-'})" for t in neighbors]
    for x, neighbors in r_edges_random.items()
}

print("Random bipartite graph")
for agent, targets in list(r_graph_structure_random.items())[:10]: 
    print(f"{agent}: {targets}")
    
    
print (" ")
print("positive ", {t for t, l in r_labels_random.items() if l == 1})
print("negative ", {t for t, l in r_labels_random.items() if l == -1})
    
graph_random['n'], graph_random['m']


Random bipartite graph
x_0: []
x_1: []
x_2: []
x_3: []
x_4: ['t_2 (+)']
x_5: ['t_2 (+)', 't_10 (-)', 't_17 (-)', 't_28 (+)']
x_6: ['t_7 (+)', 't_38 (+)']
x_7: []
x_8: []
x_9: ['t_6 (+)', 't_7 (+)', 't_17 (-)', 't_38 (+)']
 
positive  {0, 33, 2, 6, 7, 38, 9, 39, 13, 24, 28, 31}
negative  {34, 3, 35, 36, 37, 8, 10, 11, 12, 17, 26, 29}


(206, 24)

## General

In [7]:
#############
##knn##
#############
graphsRandom1 = []
graphsinfor = []
graphid = 0
for kmax1 in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    _, _, graphx_random1 = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                XRHS=X_RHS_random, 
                                                yRHS=y_RHS_random, 
                                                sLHS=s_LHS_random, 
                                                sRHS=s_RHS_random,                                        
                                                k_min=1, 
                                                k_max=kmax1)
    
    graphx_random1.update({"k_max": kmax1, "knn": "yes", "thresh": "no"})
    graphsRandom1.append(graphx_random1)
    
    ########
    avg_left_deg, avg_left_pos_deg, avg_left_neg_deg, avg_right_deg, avg_overlap, \
    avg_pos_overlap, avg_neg_overlap, only_pos, only_neg, empty_adj, unipos  = getconnectivity_info(graphx_random1["edges"], graphx_random1["labels"])

    graphsinfor.append({
        "Dataset (d)": "Student-math ("+ str(X.shape[1]) + ")",
        "kmax": kmax1, 
        "n": graphx_random1['n'],
        "m": graphx_random1['m'],
        "#+ves": len({t for t, l in graphx_random1['labels'].items() if l == 1}),
        "#-ves": len({t for t, l in graphx_random1['labels'].items() if l == -1}), 
        "avg LHSd": avg_left_deg, 
        "avg LHS+d": avg_left_pos_deg, 
        "avg LHS-d": avg_left_neg_deg,  
        "avg RHSd": avg_right_deg,         
        "avg overlap": avg_overlap,
        "avg overlap+": avg_pos_overlap,
        "avg overlap-": avg_neg_overlap,
        "only+Ns": only_pos,
        "only-Ns": only_neg,
        "emptyNs": empty_adj,
        "uni+": unipos,   
        "graphID": graphid
    })
    
    graphid += 1

######
randomgraphsinfo = pd.DataFrame(graphsinfor)   

######
np.save("../graphs/studentmath_knn_random.npy", np.array(graphsRandom1, dtype=object), allow_pickle=True)
randomgraphsinfo.to_csv("../graphs/studentmath_knn_graphsummary.npy", index=False)
######
randomgraphsinfo
 


,Dataset (d),kmax,n,m,#+ves,#-ves,avg LHSd,avg LHS+d,avg LHS-d,avg RHSd,avg overlap,avg overlap+,avg overlap-,only+Ns,only-Ns,emptyNs,uni+,graphID
0,Student-math (30),1,206,35,16,19,1.0,0.514563,0.485437,5.885714,0.037367,0.022638,0.014729,106,100,0,0,0
1,Student-math (30),2,206,38,17,21,2.0,1.038835,0.961165,10.842105,0.167274,0.097229,0.070045,60,52,0,0,1
2,Student-math (30),3,206,39,17,22,3.0,1.461165,1.538835,15.846154,0.383282,0.185982,0.197300,21,30,0,0,2
3,Student-math (30),4,206,39,17,22,4.0,2.014563,1.985437,21.128205,0.659910,0.333696,0.326214,13,14,0,0,3
4,Student-math (30),5,206,39,17,22,5.0,2.582524,2.417476,26.410256,1.014255,0.532228,0.482027,5,8,0,0,4
5,Student-math (30),6,206,39,17,22,6.0,3.053398,2.946602,31.692308,1.459531,0.742884,0.716647,5,6,0,0,5
6,Student-math (30),7,206,39,17,22,7.0,3.645631,3.354369,36.974359,1.950225,1.036088,0.914137,1,2,0,0,6
7,Student-math (30),8,206,39,17,22,8.0,4.184466,3.815534,42.256410,2.470613,1.343263,1.127350,0,1,0,0,7
8,Student-math (30),9,206,39,17,22,9.0,4.757282,4.242718,47.538462,3.082216,1.719204,1.363012,0,0,0,0,8
9,Student-math (30),10,206,39,17,22,10.0,5.271845,4.728155,52.820513,3.762160,2.094956,1.667203,0,0,0,0,9


In [8]:
#############
##thresh##
#############
graphsRandom11 = []
graphsinfor11 = []
graphid11 = 0
for t11 in [4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10]:
    _, _, graphx_random11 = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                XRHS=X_RHS_random, 
                                                yRHS=y_RHS_random,
                                                sLHS=s_LHS_random,
                                                sRHS=s_RHS_random, 
                                                threshold=t11, 
                                                topk=False, 
                                                thresh=True)
    
    graphx_random11.update({"threshold": t11, "knn": "no", "thresh": "yes"})
    graphsRandom11.append(graphx_random11)

    ########
    avg_left_deg11, avg_left_pos_deg11, avg_left_neg_deg11, avg_right_deg11, avg_overlap11, \
    avg_pos_overlap11, avg_neg_overlap11, only_pos11, only_neg11, empty_adj11, unipos11  = getconnectivity_info(graphx_random11["edges"], graphx_random11["labels"])


    graphsinfor11.append({
        "Dataset (d)": "Student-math ("+ str(X.shape[1]) + ")",
        "r": t11, 
        "n": graphx_random11['n'],
        "m": graphx_random11['m'],
        "#+ves": len({t for t, l in graphx_random11['labels'].items() if l == 1}),
        "#-ves": len({t for t, l in graphx_random11['labels'].items() if l == -1}), 
        "avg LHSd": avg_left_deg11, 
        "avg LHS+d": avg_left_pos_deg11, 
        "avg LHS-d": avg_left_neg_deg11,  
        "avg RHSd": avg_right_deg11,         
        "avg overlap": avg_overlap11,
        "avg overlap+": avg_pos_overlap11,
        "avg overlap-": avg_neg_overlap11,  
        "only+Ns": only_pos11,
        "only-Ns": only_neg11,
        "emptyNs": empty_adj11,
        "uni+": unipos11,
        "graphID": graphid11
    })
    
    graphid11 += 1

#######
randomgraphsinfo11 = pd.DataFrame(graphsinfor11)   
    
#######   
np.save("../graphs/studentmath_thresh_random.npy", np.array(graphsRandom11, dtype=object), allow_pickle=True)
randomgraphsinfo11.to_csv("../graphs/studentmath_thresh_graphsummary.npy", index=False)
#######
randomgraphsinfo11


,Dataset (d),r,n,m,#+ves,#-ves,avg LHSd,avg LHS+d,avg LHS-d,avg RHSd,avg overlap,avg overlap+,avg overlap-,only+Ns,only-Ns,emptyNs,uni+,graphID
0,Student-math (30),4.0,206,9,5,4,0.092233,0.063107,0.029126,2.111111,0.005063,0.004430,0.000633,10,5,190,1,0
1,Student-math (30),4.5,206,16,7,9,0.213592,0.111650,0.101942,2.750000,0.012368,0.006431,0.005937,14,13,174,0,1
2,Student-math (30),5.0,206,24,12,12,0.582524,0.368932,0.213592,5.000000,0.044684,0.028946,0.015738,25,19,145,0,2
3,Student-math (30),5.5,206,32,15,17,1.713592,0.966019,0.747573,11.031250,0.201349,0.116471,0.084878,24,34,97,0,3
4,Student-math (30),6.0,206,38,17,21,3.631068,2.038835,1.592233,19.684211,0.681773,0.404200,0.277573,21,27,61,0,4
5,Student-math (30),6.5,206,39,17,22,7.092233,3.902913,3.189320,37.461538,2.086657,1.221596,0.865060,8,22,33,0,5
6,Student-math (30),7.0,206,39,17,22,11.810680,6.233010,5.577670,62.384615,4.921344,2.769486,2.151857,8,23,9,0,6
7,Student-math (30),7.5,206,39,17,22,17.553398,8.956311,8.597087,92.717949,9.736879,5.188187,4.548693,2,12,3,0,7
8,Student-math (30),8.0,206,39,17,22,23.296117,11.572816,11.723301,123.051282,15.820649,8.202084,7.618565,0,8,1,0,8
9,Student-math (30),8.5,206,39,17,22,28.291262,13.674757,14.616505,149.435897,22.128439,11.164480,10.963959,1,3,0,0,9
